In [ ]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df = pd.read_csv(r"C:\Users\sjs93\Downloads\lfp_kinematics_habit_dishabit.csv")


In [13]:
# --------------------------------------------------
# UM hierarchy ranks used for relative-rank analysis
#
# These UM (urine marking) ranks were taken from:
# “Just the results from 8/30 - done during habit dishabit”
#
# We specifically used the UM hierarchy data collected
# during the Habituation/Dishabituation phase because:
# - it was temporally aligned with the recordings
# - HCO rankings were incomplete for Cage 4
# - it provided complete hierarchy information across cages
#
# Lower numerical rank = more dominant
# --------------------------------------------------

um_rank_map = {
    "1.1": 2,
    "1.2": 1,
    "1.3": 3,

    "2.1": 1,
    "2.2": 2,
    "2.3": 4,
    "2.4": 2,

    "3.1": 2,
    "3.2": 3,
    "3.3": 1,

    "4.1": 3,
    "4.3": 1,
    "4.4": 1
}

## Habituation/Dishabituation Partner Schedule

The `schedule` dictionary maps each subject mouse to:
- the repeating cagemate (`A`)
- the novel cagemate (`B`)

used during the Habituation/Dishabituation paradigm in
Phase 1 of Meghan Cum’s SocialMemoryEphys Pilot 2 experiment.

These mappings were manually generated from the experimental
trial schedule spreadsheet: https://uflorida.sharepoint.com/:x:/r/teams/Padilla-CoreanoLab/Shared%20Documents/General/Data/SocialMemoryEphysPilot2/social_mem_ephys_pilot2_schedule%201.xlsx?d=w67ee7b5b2cdf4f59835b9b96c6b1313b&csf=1&web=1&e=Eryf8m

Experimental structure:
- Exposures 1–4 (`A`) = repeating social partner
- Exposure 5 (`B`) = novel social partner

This mapping was used to:
1. identify the actual partner mouse for each sniff event
2. append partner-specific hierarchy ranks
3. compute subject-relative hierarchy relationships

Example:
- Subject `1.1`
    - repeating partner (`A`) = `1.2`
    - novel partner (`B`) = `1.3`

Relative rank was then calculated using the UM hierarchy
ranks collected during the Habituation/Dishabituation phase.

In [14]:
schedule = {

    "1.1": {"A": "1.2", "B": "1.3"},
    "1.2": {"A": "1.3", "B": "1.1"},
    "1.3": {"A": "1.1", "B": "1.2"},

    "2.1": {"A": "2.4", "B": "2.3"},
    "2.2": {"A": "2.3", "B": "2.4"},
    "2.3": {"A": "2.2", "B": "2.1"},
    "2.4": {"A": "2.1", "B": "2.2"},

    "3.1": {"A": "3.2", "B": "3.3"},
    "3.2": {"A": "3.3", "B": "3.1"},
    "3.3": {"A": "3.1", "B": "3.2"},

    "4.1": {"A": "4.3", "B": "4.4"},
    "4.4": {"A": "4.1", "B": "4.3"},
}

In [15]:
# Convert subject and partner columns to strings
#
# This ensures that:
# - subject IDs (e.g., "1.1")
# - partner labels (e.g., "A" and "B")
#
# are treated as text rather than numeric values.
#
# This step is important because the downstream:
# - um_rank_map dictionary
# - schedule dictionary
#
# use STRING keys for lookup/mapping operations.
#
# Without converting to strings, pandas may interpret:
# - subject IDs as floats
# - partner labels inconsistently
#
# which can cause dictionary mapping failures and
# produce missing (NaN) values during:
# - partner identity assignment
# - hierarchy-rank mapping
# - relative-rank calculations

df["subject"] = df["subject"].astype(str)
df["partner"] = df["partner"].astype(str)

In [16]:
# --------------------------------------------------
# Map experimental partner labels ("A" / "B")
# to the actual social partner mouse ID
#
# Each row represents a neural spectral measurement
# (e.g., coherence/power/granger) computed from
# social-investigation behavioral epochs.
#
# The original dataframe stores:
# - the subject mouse
# - whether the social interaction partner was:
#
#     "A" = repeating cagemate
#     "B" = novel cagemate
#
# This function uses the experimentally-defined
# Habituation/Dishabituation schedule to identify
# the actual cagemate mouse corresponding to each
# row.
#
# Example:
# subject = "1.1"
# partner label = "A"
#
# -> repeating cagemate = "1.2"
#
# The returned value becomes the:
#     partner_mouse
#
# column used for downstream:
# - hierarchy-rank mapping
# - relative-rank analysis
# - GLMs
# --------------------------------------------------

def get_partner_mouse(row):

    # extract subject mouse ID
    subj = row["subject"]

    # extract partner label
    # ("A" = repeating cagemate,
    #  "B" = novel cagemate)
    label = row["partner"]

    # verify:
    # - subject exists in schedule dictionary
    # - partner label is valid
    if subj in schedule and label in ["A", "B"]:

        # return actual cagemate mouse ID
        return schedule[subj][label]

    # return NaN if mapping fails
    return np.nan

In [17]:
# Apply the partner-mapping function row-by-row
#
# For each neural-analysis observation:
# - read the subject mouse ID
# - read whether the interaction occurred with:
#     "A" = repeating cagemate
#     "B" = novel cagemate
#
# The get_partner_mouse() function then uses the
# Habituation/Dishabituation schedule dictionary
# to identify the actual cagemate mouse involved
# in that observation.
#
# The resulting partner mouse ID is stored in:
#     df["partner_mouse"]
#
# Example:
# subject = "1.1"
# partner label = "A"
#
# -> repeating cagemate = "1.2"
#
# axis=1 tells pandas to apply the function
# across rows.
#
# Each row represents a neural spectral observation
# (e.g., coherence/power/granger) computed from
# social-investigation behavioral epochs during the
# Habituation/Dishabituation paradigm.

df["partner_mouse"] = df.apply(
    get_partner_mouse,
    axis=1
)

In [18]:
# --------------------------------------------------
# Append UM hierarchy ranks to the dataframe
#
# The um_rank_map dictionary contains the UM
# (urine marking) hierarchy rank assigned to
# each mouse during the Habituation/Dishabituation
# phase of the experiment.
#
# Lower numerical rank = more dominant
#
# Example:
# "1.2" -> rank 1 (more dominant)
# "1.3" -> rank 3 (less dominant)
#
# These hierarchy values are appended for:
# - the subject mouse
# - the interacting cagemate
#
# allowing downstream comparison of:
# - subject hierarchy position
# - repeating/novel cagemate hierarchy position
#
# which is later used to compute:
# - subject-relative hierarchy relationships
# - GLMs examining hierarchy-related effects on
#   neural spectral measurements
# --------------------------------------------------

# Map UM hierarchy rank for the subject mouse
#
# Example:
# subject = "1.1"
# -> subject_um_rank = 2

df["subject_um_rank"] = (
    df["subject"]
    .map(um_rank_map)
)

# Map UM hierarchy rank for the interacting cagemate
#
# Example:
# partner_mouse = "1.2"
# -> partner_um_rank = 1

df["partner_um_rank"] = (
    df["partner_mouse"]
    .map(um_rank_map)
)

In [19]:
# --------------------------------------------------
# Compute the hierarchy relationship between:
# - the subject mouse
# - the interacting cagemate
#
# This function compares the UM hierarchy ranks
# assigned to:
# - the subject mouse
# - the repeating or novel cagemate
#
# to determine whether the interacting cagemate is:
#
# - more dominant than the subject
# - less dominant than the subject
# - equal hierarchy rank
#
# IMPORTANT:
# Lower numerical rank = more dominant
#
# Example:
#
# subject_um_rank = 2
# partner_um_rank = 1
#
# Since:
#     1 < 2
#
# the interacting cagemate is MORE dominant
# than the subject.
#
# -> returns:
#    "higher_than_subject"
#
# This relative-rank classification is later used
# in downstream GLMs examining whether:
# - coherence
# - power
# - granger causality
#
# vary depending on the hierarchy relationship
# between the subject and the interacting cagemate.
# --------------------------------------------------

def relative_rank(row):

    # extract subject UM hierarchy rank
    subj = row["subject_um_rank"]

    # extract partner UM hierarchy rank
    partner = row["partner_um_rank"]

    # return NaN if either hierarchy rank is missing
    if pd.isna(subj) or pd.isna(partner):

        return np.nan

    # lower numerical rank = more dominant

    # partner is more dominant than subject
    if partner < subj:

        return "higher_than_subject"

    # partner is less dominant than subject
    elif partner > subj:

        return "lower_than_subject"

    # partner and subject share the same rank
    else:

        return "equal_rank"

In [20]:
# Apply the relative-rank classification function
# across all neural-analysis observations in the dataframe.
#
# For each observation, the function compares:
# - the subject mouse's UM hierarchy rank
# - the interacting cagemate's UM hierarchy rank
#
# and assigns a subject-relative hierarchy label:
#
# - "higher_than_subject"
#     -> interacting cagemate is more dominant
#
# - "lower_than_subject"
#     -> interacting cagemate is less dominant
#
# - "equal_rank"
#     -> subject and cagemate share the same rank
#
# The resulting hierarchy relationship is stored in:
#     df["relative_rank"]
#
# This variable is later used in downstream GLMs
# to test whether neural spectral measurements
# (e.g., coherence, power, granger causality)
# differ depending on the hierarchy relationship
# between the subject and the interacting
# repeating/novel cagemate.
#
# axis=1 tells pandas to apply the function
# row-by-row across the dataframe.

df["relative_rank"] = df.apply(
    relative_rank,
    axis=1
)

In [ ]:
import pickle

In [ ]:
with open(
    r"C:\Users\sjs93\Downloads\behavior_dicts_from_frames.pkl",
    "rb"
) as f:

    behavior_dict = pickle.load(f)

In [ ]:
df[
    [
        "partner_mouse",
        "subject_um_rank",
        "partner_um_rank",
        "relative_rank"
    ]
].isna().sum()

In [ ]:
df["relative_rank"].value_counts()

In [ ]:
df[
    [
        "subject",
        "partner",
        "partner_mouse",
        "subject_um_rank",
        "partner_um_rank",
        "relative_rank"
    ]
].drop_duplicates().sort_values(
    ["subject", "partner"]
)

In [ ]:
df["event_length"].describe()

In [ ]:
df["event"].value_counts()

In [ ]:
df.to_csv(
    r"C:\Users\sjs93\OneDrive\Documents\GitHub\diff_fam_social_memory_ephys\other_peoples_sutff\sequioa\Phase1_hab_dis_rank_GLM\lfp_kinematics_habit_dishabit_relative_rank_COMPLETE.csv",
    index=False
)

# Relative-Rank GLM Analysis

## Goal

Determine whether neural spectral measurements differ depending on the hierarchy relationship between the subject mouse and the interacting cagemate during the Habituation/Dishabituation paradigm.

Specifically, this analysis asks:

> Does coherence differ when a mouse interacts with a higher- versus lower-ranked social partner?

---

# Analysis Workflow

## Step 1 — Load Processed Neural-Behavior Data

The dataframe:

```python
lfp_kinematics_habit_dishabit.csv

In [37]:
df = pd.read_csv(
    r"C:\Users\sjs93\OneDrive\Documents\GitHub\diff_fam_social_memory_ephys\other_peoples_sutff\sequioa\Phase1_hab_dis_rank_GLM\lfp_kinematics_habit_dishabit_relative_rank_COMPLETE.csv"
)

In [38]:
df.head()

,Unnamed: 0,subject,recording,event,partner,trial,condition,event_length,distance,velocity_mouse1,...,body_angle_mouse1,body_angle_mouse2,band,metric,region_or_pair,value,partner_mouse,subject_um_rank,partner_um_rank,relative_rank
0,0,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.28054,...,2.489983,0.0,Delta,power,mPFC,17.715087,1.2,2,1,higher_than_subject
1,1,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.28054,...,2.489983,0.0,Delta,power,NAc,0.592335,1.2,2,1,higher_than_subject
2,2,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.28054,...,2.489983,0.0,Delta,power,MD,-27.343484,1.2,2,1,higher_than_subject
3,3,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.28054,...,2.489983,0.0,Delta,power,BLA,-37.703427,1.2,2,1,higher_than_subject
4,4,1.1,11_cage_p1_merged.rec,exp1 sniff,A,0,cagemate,3656,275.719717,0.28054,...,2.489983,0.0,Delta,power,vHPC,41.527165,1.2,2,1,higher_than_subject


In [39]:
df.columns

Index(['Unnamed: 0', 'subject', 'recording', 'event', 'partner', 'trial',
       'condition', 'event_length', 'distance', 'velocity_mouse1',
       'velocity_mouse2', 'moving_angle_mouse1', 'moving_angle_mouse2',
       'front_orientation_mouse1', 'front_orientation_mouse2',
       'rump_orientation_mouse1', 'rump_orientation_mouse2',
       'body_angle_mouse1', 'body_angle_mouse2', 'band', 'metric',
       'region_or_pair', 'value', 'partner_mouse', 'subject_um_rank',
       'partner_um_rank', 'relative_rank'],
      dtype='object')

In [40]:
df.shape

(340375, 27)

In [41]:
df["metric"].unique()

array(['power', 'coherence', 'granger'], dtype=object)

In [42]:
df["relative_rank"].value_counts()

relative_rank
higher_than_subject    174125
lower_than_subject     149450
equal_rank              16800
Name: count, dtype: int64

In [43]:
df["region_or_pair"].unique()

array(['mPFC', 'NAc', 'MD', 'BLA', 'vHPC', 'mPFC_NAc', 'mPFC_MD',
       'mPFC_BLA', 'mPFC_vHPC', 'NAc_MD', 'NAc_BLA', 'NAc_vHPC', 'MD_BLA',
       'MD_vHPC', 'BLA_vHPC', 'mPFC_to_NAc', 'mPFC_to_MD', 'mPFC_to_BLA',
       'mPFC_to_vHPC', 'NAc_to_mPFC', 'NAc_to_MD', 'NAc_to_BLA',
       'NAc_to_vHPC', 'MD_to_mPFC', 'MD_to_NAc', 'MD_to_BLA',
       'MD_to_vHPC', 'BLA_to_mPFC', 'BLA_to_NAc', 'BLA_to_MD',
       'BLA_to_vHPC', 'vHPC_to_mPFC', 'vHPC_to_NAc', 'vHPC_to_MD',
       'vHPC_to_BLA'], dtype=object)

In [44]:
df["band"].unique()

array(['Delta', 'Theta', 'Beta', 'Low gamma', 'High gamma'], dtype=object)

In [45]:
# Subset dataframe to include only coherence
# measurements for the initial GLM analyses.
coh_df = df[
    df["metric"] == "coherence"
].copy()

In [46]:
#inspect counts of observations across relative rank and brain region pairs
coh_df.groupby(
    [
        "relative_rank",
        "band"
    ]
).size()

relative_rank        band      
equal_rank           Beta           960
                     Delta          960
                     High gamma     960
                     Low gamma      960
                     Theta          960
higher_than_subject  Beta          9950
                     Delta         9950
                     High gamma    9950
                     Low gamma     9950
                     Theta         9950
lower_than_subject   Beta          8540
                     Delta         8540
                     High gamma    8540
                     Low gamma     8540
                     Theta         8540
dtype: int64

In [47]:
#Inspect counts of observations across relative rank and brain region pairs
coh_df.groupby(
    [
        "region_or_pair",
        "relative_rank"
    ]
).size()

region_or_pair  relative_rank      
BLA_vHPC        equal_rank              480
                higher_than_subject    4975
                lower_than_subject     4270
MD_BLA          equal_rank              480
                higher_than_subject    4975
                lower_than_subject     4270
MD_vHPC         equal_rank              480
                higher_than_subject    4975
                lower_than_subject     4270
NAc_BLA         equal_rank              480
                higher_than_subject    4975
                lower_than_subject     4270
NAc_MD          equal_rank              480
                higher_than_subject    4975
                lower_than_subject     4270
NAc_vHPC        equal_rank              480
                higher_than_subject    4975
                lower_than_subject     4270
mPFC_BLA        equal_rank              480
                higher_than_subject    4975
                lower_than_subject     4270
mPFC_MD         equal_rank              

Initial exploratory analysis:

coherence ~ relative_rank * frequency_band + event_length + velocity

• Exploratory fixed-effect OLS models were run separately for each region pair
• Recording session random effects were NOT yet included
• Current p-values should therefore be interpreted as preliminary/exploratory
• Future analyses will implement mixed-effects models:

coherence ~ relative_rank * frequency_band + event_length + velocity + (1 | recording_session)

to account for repeated observations within recording sessions


In [48]:
coh_df = df[
    df["metric"] == "coherence"
].copy()

In [49]:
region_pairs = (
    coh_df["region_or_pair"]
    .unique()
)

In [50]:
results = []

for region in region_pairs:

    test_df = coh_df[
        coh_df["region_or_pair"] == region
    ].copy()

    model = smf.ols(
        formula=
        "value ~ relative_rank * band + velocity_mouse1 + event_length",
        data=test_df
    ).fit()

    pval = model.f_pvalue

    results.append({
        "region_pair": region,
        "model_pvalue": pval,
        "r_squared": model.rsquared
    })

In [51]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    "model_pvalue"
)

,region_pair,model_pvalue,r_squared
4,NAc_MD,2.815326e-75,0.049590
7,MD_BLA,8.546856e-66,0.042852
5,NAc_BLA,2.391905e-60,0.043956
0,mPFC_NAc,5.296310e-37,0.028839
6,NAc_vHPC,5.171057e-36,0.034693
9,BLA_vHPC,5.975865e-32,0.033333
1,mPFC_MD,3.297570e-29,0.023086
2,mPFC_BLA,7.807292e-29,0.023101
3,mPFC_vHPC,2.931491e-27,0.024442
8,MD_vHPC,4.483315e-22,0.022566


In [52]:
# copy results dataframe
table_df = results_df.copy()

# format p-values
table_df["p_value"] = table_df["model_pvalue"].apply(
    lambda x: "< 0.001" if x < 0.001 else f"{x:.3f}"
)

# significance stars
def stars(p):

    if p < 0.001:
        return "***"

    elif p < 0.01:
        return "**"

    elif p < 0.05:
        return "*"

    else:
        return ""

table_df["sig"] = (
    table_df["model_pvalue"]
    .apply(stars)
)

# optional prettier region names
table_df["region_pair"] = (
    table_df["region_pair"]
    .str.replace("_", " ↔ ")
)

# keep only display columns
table_df = table_df[
    [
        "region_pair",
        "p_value",
        "sig",
        "r_squared"
    ]
]

# sort by significance
table_df = table_df.sort_values(
    "r_squared",
    ascending=False
)

table_df

,region_pair,p_value,sig,r_squared
4,NAc ↔ MD,< 0.001,***,0.049590
5,NAc ↔ BLA,< 0.001,***,0.043956
7,MD ↔ BLA,< 0.001,***,0.042852
6,NAc ↔ vHPC,< 0.001,***,0.034693
9,BLA ↔ vHPC,< 0.001,***,0.033333
0,mPFC ↔ NAc,< 0.001,***,0.028839
3,mPFC ↔ vHPC,< 0.001,***,0.024442
2,mPFC ↔ BLA,< 0.001,***,0.023101
1,mPFC ↔ MD,< 0.001,***,0.023086
8,MD ↔ vHPC,< 0.001,***,0.022566


In [53]:
test_df = coh_df[
    coh_df["region_or_pair"] == "MD_BLA"
].copy()

In [54]:
test_df["recording"].value_counts()

recording
23_nov_p1_merged.rec     750
21_cage_p1_merged.rec    690
44_nov_p1_merged.rec     600
41_nov_p1_merged.rec     580
32_nov_p1_merged.rec     575
21_nov_p1_merged.rec     565
11_nov_p1_merged.rec     520
32_cage_p1_merged.rec    455
41_cage_p1_merged.rec    455
22_nov_p1_merged.rec     440
12_cage_p1_merged.rec    435
33_nov_p1_merged.rec     390
24_cage_p1_merged.rec    385
31_cage_p1_merged.rec    360
23_cage_p1_merged.rec    335
12_nov_p1_merged.rec     325
22_cage_p1_merged.rec    325
13_nov_p1_merged.rec     280
31_nov_p1_merged.rec     275
13_cage_p1_merged.rec    275
11_cage_p1_merged.rec    245
24_nov_p1_merged.rec     195
44_cage_p1_merged.rec    190
33_cage_p1_merged.rec     80
Name: count, dtype: int64

In [57]:
test_df = coh_df[
    coh_df["region_or_pair"] == "MD_BLA"
].copy()

# VERY IMPORTANT
test_df = test_df.reset_index(
    drop=True
)

In [59]:
mixed_model = smf.mixedlm(
    formula=
    "value ~ relative_rank * band + velocity_mouse1 + event_length",

    data=test_df,

    groups=test_df["recording"]
)

IndexError: index 8120 is out of bounds for axis 0 with size 8120